# 6. 실제 데이터 실습: 2026 교정통계연보

웹 강의자료: [https://statjeong.github.io/moj-python-lecture/06_yearbook.html](https://statjeong.github.io/moj-python-lecture/06_yearbook.html)

이 장에서는 법무부가 공개한 『2026 교정통계연보』의 표를 그대로 옮긴 자료로, 1~5장에서 배운 방법을 실제 숫자에 적용합니다. 수용률 계산 함수, 병합, 반복문, 증가율, 그래프를 다시 씁니다.

## 6.1 자료 소개

| 파일 | 연보 표 | 쪽 | 내용 |
|---|---|---|---|
| `capacity_1991_2025.csv` | 표Ⅱ-13 | 120~121 | 연도별 수용정원, 1일 평균 수용인원, 수용률 (35행) |
| `staff_2016_2025.csv` | 표Ⅰ-15 | 38 | 교정공무원 정원, 현원 (10행) |
| `offense_2016_2025.csv` | 표Ⅱ-9 | 110 | 수형자 죄명별 인원 (10행) |
| `age_2016_2025.csv` | 표Ⅱ-8 | 108 | 수형자 연령별 인원 (10행) |

출처: 이 자료는 『2026 교정통계연보』(법무부 교정본부, 2026) 38쪽, 108쪽, 110쪽, 120~121쪽에서 전재하였습니다. 원문은 교정본부 누리집(www.corrections.go.kr) 알림마당 > 교정통계연보에서 내려받을 수 있습니다.

PDF의 표를 CSV로 옮긴 뒤, 표 안의 합계가 맞는지(예: 죄명별 인원의 합 = 계)를 모두 확인했습니다. 실무에서 PDF나 한글 파일의 표를 옮겨 쓸 때도 이런 확인을 먼저 합니다.

## 6.2 준비

In [ ]:
!apt-get -qq install -y fonts-nanum > /dev/null

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

주소 = "https://raw.githubusercontent.com/StatJeong/moj-python-lecture/main/data/yearbook/"
수용 = pd.read_csv(주소 + "capacity_1991_2025.csv")
직원 = pd.read_csv(주소 + "staff_2016_2025.csv")
죄명 = pd.read_csv(주소 + "offense_2016_2025.csv")
연령 = pd.read_csv(주소 + "age_2016_2025.csv")

## 6.3 수용률 35년 추이

In [ ]:
수용.head()

In [ ]:
수용.tail()

### 연보의 수용률을 직접 다시 계산하기

1장에서 만든 수용률 함수로 35년치 수용률을 다시 계산하고, 연보에 적힌 값과 같은지 확인합니다. 함수에 열을 통째로 넣으면 모든 행이 한 번에 계산됩니다.

In [ ]:
def 수용률계산(현원, 정원):
    return round(현원 / 정원 * 100, 1)

수용["수용률_계산"] = 수용률계산(수용["1일평균수용인원"], 수용["수용정원"])
수용[["연도", "수용률", "수용률_계산"]].tail()

In [ ]:
print((수용["수용률_계산"] == 수용["수용률"]).all())

35개 연도 모두 같습니다. 공개된 통계라도 이렇게 한 번 다시 계산해 보면, 자료를 옮기다 생긴 실수나 정의의 차이를 찾을 수 있습니다.

### 가장 높았던 해, 정원 이내였던 해

In [ ]:
수용.loc[수용["수용률"].idxmax()]

In [ ]:
수용[수용["수용률"] < 100][["연도", "수용정원", "1일평균수용인원", "수용률"]]

`idxmax()`는 가장 큰 값이 있는 행의 번호를 돌려주고, `loc[]`은 그 번호의 행을 꺼냅니다. 35년 동안 수용률이 100% 아래였던 해는 1992년, 2006년, 2012년 세 해뿐입니다.

### 그래프로 보기

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(수용["연도"], 수용["수용률"], marker="o")
plt.axhline(100, color="gray", linestyle="--")
plt.title("교정시설 수용률 (1991~2025년)")
plt.xlabel("연도")
plt.ylabel("수용률(%)")
plt.show()

수용률이 가장 높았던 해는 2003년(132.9%)인데, 1일 평균 수용인원이 가장 많았던 해는 1999년(68,087명)입니다. 정원과 수용인원을 함께 그려 보면 이유가 보입니다.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(수용["연도"], 수용["수용정원"], label="수용정원")
plt.plot(수용["연도"], 수용["1일평균수용인원"], label="1일 평균 수용인원")
plt.axvline(2003, color="gray", linestyle=":")
plt.title("수용정원과 1일 평균 수용인원 (1991~2025년)")
plt.xlabel("연도")
plt.ylabel("인원(명)")
plt.legend()
plt.show()

2003년에 수용인원은 조금 줄었는데 정원이 58,440명에서 44,350명으로 크게 줄었습니다. 연보(121쪽)에 따르면 2002년 12월 「법무시설기준규칙」의 혼거실 1인당 수용면적 기준이 1.65㎡에서 2.48㎡로 넓어졌기 때문입니다. 수용률의 분모인 정원의 기준이 바뀌었으므로, 2003년 전후의 수용률은 같은 잣대로 비교하기 어렵습니다. 숫자를 비교하기 전에 그 숫자의 정의가 바뀐 적이 있는지 확인해야 합니다.

## 6.4 교정공무원 1인당 수용인원 (병합)

In [ ]:
직원

수용 표와 직원 표를 `연도` 기준으로 합칩니다.

In [ ]:
합친표 = pd.merge(수용, 직원, on="연도")
합친표["1인당수용인원"] = (합친표["1일평균수용인원"] / 합친표["현원"]).round(2)
합친표[["연도", "1일평균수용인원", "현원", "1인당수용인원"]]

수용 표는 35행인데 합친 결과는 10행입니다. 직원 표에 2016년부터의 값만 있어서, 짝이 없는 1991~2015년은 빠졌습니다(2장 2.11절).

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(합친표["연도"], 합친표["1인당수용인원"], marker="o")
plt.title("교정공무원 1인당 1일 평균 수용인원")
plt.xlabel("연도")
plt.ylabel("명")
plt.show()

### 증가율 함수

2016년과 2025년을 비교하는 증가율 함수를 만듭니다. `iloc[0]`은 첫 행, `iloc[-1]`은 마지막 행입니다(1장 리스트의 위치와 같은 방식).

In [ ]:
def 증가율(처음, 나중):
    return round((나중 - 처음) / 처음 * 100, 1)

처음 = 합친표.iloc[0]
나중 = 합친표.iloc[-1]
print("1일 평균 수용인원:", 증가율(처음["1일평균수용인원"], 나중["1일평균수용인원"]), "%")
print("수용정원:", 증가율(처음["수용정원"], 나중["수용정원"]), "%")
print("교정공무원 현원:", 증가율(처음["현원"], 나중["현원"]), "%")

2016년부터 2025년까지 1일 평균 수용인원은 12.7% 늘었는데, 수용정원은 8.6%, 교정공무원 현원은 6.5% 늘었습니다. 그래서 1인당 수용인원이 3.59명에서 3.80명으로 늘었습니다. 이 값은 현원 기준이고, 행정·의료 인력까지 모두 포함한 수치입니다.

## 6.5 죄명별 구성 변화

In [ ]:
죄명

먼저 옮긴 표가 맞는지 확인합니다. 죄명별 인원을 더하면 '계'와 같아야 합니다.

In [ ]:
죄명목록 = ["절도", "폭력행위등처벌법", "사기횡령", "성폭력", "마약류", "과실범", "강도", "살인", "기타"]
print((죄명[죄명목록].sum(axis=1) == 죄명["계"]).all())

`sum(axis=1)`은 행마다 옆으로 더하라는 뜻입니다.

### 구성비

반복문으로 죄명마다 구성비(%)를 계산해 새 표를 만듭니다.

In [ ]:
구성비 = pd.DataFrame({"연도": 죄명["연도"]})
for 이름 in 죄명목록:
    구성비[이름] = (죄명[이름] / 죄명["계"] * 100).round(1)
구성비

In [ ]:
plt.figure(figsize=(8, 4))
for 이름 in ["사기횡령", "마약류", "절도"]:
    plt.plot(구성비["연도"], 구성비[이름], marker="o", label=이름)
plt.title("주요 죄명의 구성비 변화 (2016~2025년)")
plt.xlabel("연도")
plt.ylabel("구성비(%)")
plt.legend()
plt.show()

### 죄명별 증가율

6.4절의 증가율 함수를 반복문으로 죄명마다 부르고, 결과를 표로 모읍니다.

In [ ]:
결과 = []
for 이름 in 죄명목록:
    결과.append({
        "죄명": 이름,
        "2016년": 죄명[이름].iloc[0],
        "2025년": 죄명[이름].iloc[-1],
        "증가율": 증가율(죄명[이름].iloc[0], 죄명[이름].iloc[-1]),
    })

증감 = pd.DataFrame(결과).sort_values("증가율")
증감

In [ ]:
plt.figure(figsize=(7, 4))
plt.barh(증감["죄명"], 증감["증가율"])
plt.axvline(0, color="gray")
plt.title("수형자 죄명별 인원 증가율 (2016년 대비 2025년)")
plt.xlabel("증가율(%)")
plt.show()

마약류(121.6%)와 과실범(95.3%)이 크게 늘었고, 강도(-43.6%)와 절도(-32.8%)는 줄었습니다. 사기·횡령은 36.3% 늘어 2025년 구성비가 24.4%로 가장 큽니다.

죄명별 표의 '계'(2025년 43,074명)는 6.3절의 1일 평균 수용인원에 들어 있는 수형자 수(2025년 40,564명, 표Ⅱ-1)와 다릅니다. 두 표의 집계 기준이 다른 것으로 보이며 연보에 기준이 따로 적혀 있지 않습니다. 그래서 이 절에서는 인원보다 구성비와 증가율로 비교했습니다.

## 6.6 고령 수형자

In [ ]:
연령["60세이상_비중"] = (연령["60세이상"] / 연령["계"] * 100).round(1)
연령[["연도", "60세이상", "60세이상_비중"]]

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(연령["연도"], 연령["60세이상_비중"])
plt.title("수형자 중 60세 이상 비중")
plt.xlabel("연도")
plt.ylabel("비중(%)")
plt.show()

In [ ]:
print("60세 이상 수형자 증가율:", 증가율(연령["60세이상"].iloc[0], 연령["60세이상"].iloc[-1]), "%")

60세 이상 수형자는 2016년 3,841명에서 2025년 7,789명으로 두 배 넘게 늘었고, 비중은 10.5%에서 18.1%가 되었습니다.

## 6.7 AI와 함께 해보기

5장의 방법으로 AI에게 코드를 받아 해결하고, 받은 코드를 한 줄씩 묻고 결과를 확인하세요. 정답 보기에는 확인용 코드가 있습니다.

1. 죄명 표를 "연도, 죄명, 인원" 세 열짜리 긴 표로 바꾸고, 9개 죄명의 인원 추이를 한 그래프에 그리세요. AI가 `melt`라는 함수를 쓰면 "melt가 표를 어떻게 바꾸나요?"라고 물어보세요.

In [ ]:
# AI에게 받은 코드를 붙여 넣어 실행해 보세요

2. 6.3절의 수용률 그래프에 2003년 위치에 세로선을 긋고 "수용면적 기준 변경"이라는 글자를 넣어 달라고 요청하세요.

In [ ]:
# AI에게 받은 코드를 붙여 넣어 실행해 보세요

3. 시작 연도, 끝 연도, 열 이름 목록을 넣으면 열마다 증가율을 표로 돌려주는 함수를 만들어 달라고 요청하세요(5.6절의 요청문 틀 활용). 2016년, 2025년으로 넣었을 때 6.5절의 증가율과 같은지 확인한 뒤, 2021년과 2025년을 비교해 보세요.

In [ ]:
# AI에게 받은 함수를 붙여 넣어 실행해 보세요

4. 이 장의 결과를 바탕으로 AI에게 "교정시설 현황 요약" 다섯 문장을 써 달라고 요청하세요. 받은 문장의 숫자를 아래 값과 하나씩 대조합니다.

| 항목 | 값 | 확인한 곳 |
|---|---|---|
| 2025년 수용률 | 125.8% | 6.3절 |
| 2016년 대비 1일 평균 수용인원 증가율 | 12.7% | 6.4절 |
| 2025년 교정공무원 1인당 수용인원 | 3.80명 | 6.4절 |
| 2016년 대비 마약류 수형자 증가율 | 121.6% | 6.5절 |
| 2025년 60세 이상 수형자 비중 | 18.1% | 6.6절 |